# 🎙️ Speech Intelligence Engine — Google Colab GPU Trainer

This notebook offloads heavy **Speech Foundation Model** training, SSL feature extraction (WavLM / HuBERT), and Speech-LLM adapter fine-tuning to Google Colab GPUs (T4 / L4 / A100), persisting all checkpoints directly to your 5TB **Google Drive** (`rakshithgoud453@gmail.com`).

## Step 1: Check GPU & Mount Google Drive (5TB Storage)

In [21]:
# Check available GPU resources
!nvidia-smi

# Mount 5TB Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')

for path in [DRIVE_BASE_PATH, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"✓ Ready directory on Google Drive: {path}")

Fri Oct  2 04:46:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install ML & Audio Dependencies (Colab-Optimized)

In [22]:
# Install extra NLP, Audio & ML packages without overriding Colab's pre-installed CUDA PyTorch
!pip install -q transformers datasets peft accelerate bitsandbytes librosa soundfile wandb pyannote.audio faster-whisper hdbscan

## Step 3: Clone / Sync Repository Code

In [23]:
# Codebase setup in Colab workspace
import os

REPO_URL = "https://github.com/rakshithgoud453/speech_intelligence_engine.git"
WORKSPACE_DIR = "/content/speech_intelligence_engine"

%cd /content
if os.path.exists(WORKSPACE_DIR):
    %cd {WORKSPACE_DIR}
    !git pull
else:
    !git clone {REPO_URL} {WORKSPACE_DIR}
    %cd {WORKSPACE_DIR}

print(f"✓ Currently operating in: {os.getcwd()}")

/content
/content/speech_intelligence_engine
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.56 KiB | 227.00 KiB/s, done.
From https://github.com/rakshithgoud453/speech_intelligence_engine
   fc1632c..bafcbfc  master     -> origin/master
Updating fc1632c..bafcbfc
Fast-forward
 notebooks/speech_intelligence_colab_trainer.ipynb | 213 +++++++++++++---------
 1 file changed, 123 insertions(+), 90 deletions(-)
✓ Currently operating in: /content/speech_intelligence_engine


## Step 4: SSL Layer Representation Extraction (WavLM / HuBERT)

In [24]:
import torch
import torchaudio
from transformers import WavLMModel, AutoFeatureExtractor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Pre-trained SSL Model (microsoft/wavlm-base-plus)
MODEL_ID = "microsoft/wavlm-base-plus"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
ssl_model = WavLMModel.from_pretrained(MODEL_ID, output_hidden_states=True).to(device)
ssl_model.eval()

def extract_ssl_representations(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    inputs = feature_extractor(waveform.squeeze(0).numpy(), sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = ssl_model(**inputs)
        hidden_states = outputs.hidden_states

    # Layers 4-8: Prosodic/Phonetic representations
    prosodic_rep = torch.stack(hidden_states[4:9]).mean(dim=0)
    # Layers 9-11: Semantic representations
    semantic_rep = torch.stack(hidden_states[9:12]).mean(dim=0)

    return {
        "prosodic_embedding": prosodic_rep.cpu().numpy(),
        "semantic_embedding": semantic_rep.cpu().numpy()
    }

print("✓ SSL Feature Extraction Pipeline Loaded Successfully.")

ModuleNotFoundError: Could not import module 'WavLMModel'. Are this object's requirements defined correctly?

## Step 5: Checkpoint Saver Utility

In [ ]:
import os
import time
import torch

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)

    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)

    print(f"✓ Saved Checkpoint to Google Drive (5TB): {save_path}")

print(f"✓ Checkpoints directory configured: {CHECKPOINT_DIR}")

## Step 6: Batch Audio Ingestion & Model Checkpoint Execution Loop 🚀 (Self-Contained & Auto-Logging)

In [ ]:
import sys
import glob
import json
import os
import time
import traceback
import soundfile as sf
import numpy as np
import torch

# 1. Ensure Google Drive Output Directories
DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')
for p in [CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
    os.makedirs(p, exist_ok=True)

# 2. Local Saver Fallback
def local_save_checkpoint(model, step, metrics):
    checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
    save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'metrics': metrics
    }, save_path)
    print(f"✓ Saved Checkpoint to 5TB Google Drive: {save_path}")

# 3. Execute Loop with Automated Error Logging
try:
    WORKSPACE_DIR = "/content/speech_intelligence_engine"
    for p in [WORKSPACE_DIR, os.path.dirname(WORKSPACE_DIR)]:
        if p not in sys.path:
            sys.path.insert(0, p)

    try:
        from speech_intelligence_engine import SpeechLearningEngine
    except ImportError:
        from engine import SpeechLearningEngine

    try:
        from speech_intelligence_engine.representation.projector import SpeechLLMProjector
    except ImportError:
        from representation.projector import SpeechLLMProjector

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Initializing Engine & Projector on device: {device}...")

    engine = SpeechLearningEngine(model_type="wavlm", replay_capacity=1000)
    projector = SpeechLLMProjector(input_dim=768, hidden_dim=2048, output_dim=4096).to(device)

    audio_files = glob.glob(os.path.join(DATASET_DIR, "*.wav")) + glob.glob(os.path.join(DATASET_DIR, "*.mp3"))

    if not audio_files:
        print(f"No audio files in Google Drive datasets. Generating benchmark audio sample...")
        sample_wav = os.path.join(DATASET_DIR, "sample_speech_benchmark.wav")
        t = np.linspace(0, 3.0, int(16000 * 3.0), endpoint=False)
        sig = (0.5 * np.sin(2 * np.pi * 180 * t) + 0.2 * np.sin(2 * np.pi * 360 * t)).astype(np.float32)
        sf.write(sample_wav, sig, 16000)
        audio_files = [sample_wav]

    for i, audio_path in enumerate(audio_files, 1):
        print(f"\n[Processing {i}/{len(audio_files)}]: {audio_path}")
        res = engine.assimilate(audio_path, recording_id=os.path.basename(audio_path))

        metrics = {
            "episodes": res.num_episodes,
            "voiced_segments": res.num_segments,
            "novelty_rate": res.novelty_rate,
            "duration_sec": res.duration_sec
        }
        local_save_checkpoint(projector, step=i, metrics=metrics)

    status_payload = {
        "status": "COMPLETED",
        "total_episodes_learned": engine.get_state().total_episodes,
        "prototypes_discovered": engine.get_state().total_prototypes,
        "checkpoint_directory": CHECKPOINT_DIR,
        "last_updated": time.ctime()
    }

    status_file = os.path.join(OUTPUT_DIR, "status.json")
    with open(status_file, "w") as f:
        json.dump(status_payload, f, indent=2)

    print(f"\n✨ Batch processing complete!")
    print(f"📁 Checkpoint saved to Google Drive: {CHECKPOINT_DIR}")
    print(f"📄 Status summary written to: {status_file}")

except Exception as e:
    err_trace = traceback.format_exc()
    print(f"\n❌ EXECUTION ERROR IN STEP 6:\n{err_trace}")
    error_log_path = os.path.join(OUTPUT_DIR, "error_log.txt")
    with open(error_log_path, "w") as f:
        f.write(err_trace)
    print(f"📄 Full Error Traceback written to Google Drive: {error_log_path}")
    raise e